In [1]:
import sys
import numpy as np

sys.path.append("../../../src/")
from Rain.Rain import Rain
sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-04 08:42:34.995613: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-04 08:42:35.746608: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {}
      
    },
  "temp_data_path": "../../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-04 08:42:36,744 [DEBUG] [Rain] Rain is initialized
2023-07-04 08:42:36,746 [DEBUG] [Provisioner] Creating coordinator
2023-07-04 08:42:36,748 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:42:36,749 [DEBUG] [Coordinator Ambassador] Coordinator Ambassador is initialized
2023-07-04 08:42:36,750 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-04 08:42:36,751 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/coord/
2023-07-04 08:42:36,753 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-04 08:42:36,754 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:42:36,756 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/
2023-07-04 08:42:36,757 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-04 08:42:36,764 [DEBUG] [Rain] Creating workers
2023-07-04 08:42:36,771 [INFO] [Provisioner] provisioner is serving
2023-07-04 08:42:36,771 [DEBUG] [Provisioner] Starting coordinator
2023-07-04 08:42:36,773 [INFO] [Coordinator] coordinator is serving
2023-07-04 08:42:36,774 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-04 08:42:36,778 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:42:36,779 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-04 08:42:36,780 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-04 08:42:36,781 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:42:36,784 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-04 08:42:36,785 [DEBUG] [TemporaryFilesManager] Creating temporary directory ../../../..//RainData/worker/
2023-07-04 08:42:36,787 [INFO] [W

140/157 [=========================>....] - ETA: 0s - loss: 0.7298 - accuracy: 0.7689sending data to coordinator


2023-07-04 08:42:55,356 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
2023-07-04 08:42:55,361 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2


157/157 [==============================] - 3s 8ms/step - loss: 0.6912 - accuracy: 0.7820


2023-07-04 08:42:55,450 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
2023-07-04 08:42:55,452 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3
2023-07-04 08:42:55,455 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
2023-07-04 08:42:55,457 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1


...

2023-07-04 08:43:03,318 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 2
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 2
2023-07-04 08:43:03,323 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to coordinator
157/157 [==============================] - 2s 8ms/step - loss: 0.2531 - accuracy: 0.9233


2023-07-04 08:43:03,467 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1


sending data to coordinator


DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:43:03,470 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
2023-07-04 08:43:03,625 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 08:43:03,636 [DEBUG] [DeepLearning] Asynchronous update is done by worker 3
DEBUG:DeepLearning:Asynchronous update is done by worker 3
2023-07-04 08:43:03,643 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-04 08:43:03,663 [DEBUG] 

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 1ms/step - loss: 0.1489 - accuracy: 0.9554

Test accuracy: 95.5%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-04 08:43:04,120 [DEBUG] [Rain] Creating workers
DEBUG:Rain:Creating workers
2023-07-04 08:43:04,123 [INFO] [Provisioner] provisioner is serving
INFO:Provisioner:provisioner is serving
2023-07-04 08:43:04,125 [DEBUG] [Provisioner] Starting coordinator
DEBUG:Provisioner:Starting coordinator
2023-07-04 08:43:04,129 [INFO] [Coordinator] coordinator is serving
INFO:Coordinator:coordinator is serving
2023-07-04 08:43:04,130 [DEBUG] [Coordinator] sending the num of workers to the provisioner
DEBUG:Coordinator:sending the num of workers to the provisioner
2023-07-04 08:43:04,134 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
DEBUG:Provisioner:Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-04 08:43:04,136 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
DEBUG:Coordinator:sent Success receiving the number of workers to the provisioner
2023-07-04 08:43:04,

157/157 [==============================] - 3s 8ms/step - loss: 0.2128 - accuracy: 0.9380


2023-07-04 08:43:23,277 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1


sending data to coordinator
153/157 [============================>.] - ETA: 0s - loss: 0.2062 - accuracy: 0.9372

DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:43:23,282 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_1_trained.pkl from worker1


...

DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/2_3_trained.pkl from worker2


157/157 [==============================] - 3s 8ms/step - loss: 0.1588 - accuracy: 0.9532


2023-07-04 08:43:30,962 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 1
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 1
2023-07-04 08:43:30,963 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/1_3_trained.pkl from worker1
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/1_3_trained.pkl from worker1
2023-07-04 08:43:30,966 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker 3
DEBUG:DividerAmbassador:divider received: Executed! after executing the model on worker 3
2023-07-04 08:43:30,968 [DEBUG] [DividerAmbassador] divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3
DEBUG:DividerAmbassador:divider begins downloading ../../../..//RainData/divider/3_3_trained.pkl from worker3


sending data to coordinator
sending data to coordinator


2023-07-04 08:43:31,193 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-04 08:43:31,224 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-04 08:43:31,226 [DEBUG] [DividerAmbassador] Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
DEBUG:DividerAmbassador:Downloaded ../../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-04 08:43:31,253 [DEBUG] [DeepLearning] Iteration 3/3 complete.
DEBUG:DeepLearning:Iteration 3/3 complete.
2023-07-04 08:43:31,255 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
DEBUG:DividerAmbassador:divider ambassador stopped serving
2023-07-04 08:

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 2ms/step - loss: 0.0987 - accuracy: 0.9705

Test accuracy: 97.0%


2023-07-04 08:45:07,408 [DEBUG] [Coordinator] coordinator is sending workers info to divider
DEBUG:Coordinator:coordinator is sending workers info to divider
2023-07-04 08:45:07,440 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
2023-07-04 08:45:07,440 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 1
INFO:Worker_50152:Running the worker with id: 2 on iteration: 1
2023-07-04 08:45:07,442 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
2023-07-04 08:45:07,442 [INFO] [Worker_50153] Running the worker with id: 3 on iteration: 1
INFO:Worker_50153:Running the worker with id: 3 on iteration: 1
2023-07-04 08:45:07,467 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-04 08:45:07,468 [INFO] [Worker_50151] Running the worker with id: 1 on iteration: 2
2023-07-04 08:45:07,467 [INFO] [Worker_50152] Running the worker with id: 2 on iteration: 2
2023-07-04 08:45:07,468 [INFO] [Worker_50151] Running the worker with 

sending data to coordinator
sending data to coordinator
sending data to coordinator
